# Solution 6 — RAG generation WITH the reranker — corrected

`solution_6_reranked_generation.ipynb` with paths fixed so it runs from the repo, from Colab, or
anywhere with network access, and with the defects below repaired.

| | Condition |
|---|---|
| **C1** | MSA query, e5 retrieval (ceiling) |
| **C2** | Darija query, e5 retrieval (mismatch) |
| **C5** | Darija query, e5 retrieval **+ bge-reranker-v2-m3** (the new condition) |
| **C4** | gold passage given directly (oracle) |

## Defects found and fixed

The notebook had never been run — every cell was output-free — so these were found by reading it
and by testing the pieces against your actual data.

**1. Paths.** It opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory; in this
repo those are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and it ended on a bare `from google.colab import files`, which
raises anywhere else. All resolved now: repo → working dir → GitHub, with the source printed.

**2. `prior_run_csv` pointed at a file that does not exist.** `generation_n200_raw.csv` is nowhere
in the repo or in git history. The actual earlier run is `results/generation_local_raw.csv`. The
old value failed the `os.path.exists` check silently, so the "combined" validation sheet would
have quietly covered **one** run while printing that it covers both. Now it resolves the real
file, and says loudly which runs made it into the sheet.

**3. `parse_judge` drops the correctness verdict when both appear on one line.** The loop used
`if "مدعوم" … elif "مطابق" …`, so a reply of `مدعوم: نعم مطابق: نعم` parses as
*(faithful=1, correct=0)* — a correct answer scored wrong. Tested against 6 realistic judge
replies: 2 parse incorrectly. Now parsed per field, with a regression suite in the notebook.

**4. Unparseable judge output was indistinguishable from a genuine negative.** Anything
unrecognised returned `(0, 0)`, counted as unfaithful *and* incorrect with nothing in the output to
show it. Now recorded as `judge_parsed` and reported as a rate.

**5. `stratified_sample(df, 50, …)` returns 48 rows, not 50.** `n_per = 50 // 4 = 12`, twelve per
condition, so the combined sheet is 96 rows while the notebook announces 100. Now it tops the
sample up to the requested size.

**6. Unresolvable qids produce silently blank labeling rows.** `qa_lookup.get(r["qid"], {})` yields
empty strings for question, gold answer and query — a labeler receives rows with nothing to judge
and no warning. Verified: with a lookup miss, **48 of 48 rows come out blank**. Now counted, warned
about, and dropped from the sheet.

**7. The sheet tells you to run a script that does not exist.** The final cell instructs
`python validate_judge.py score --sheet …`; there is no `validate_judge.py` in the repo. Replaced
with a description of the sheet's columns and what to do with it.

**8. Faithfulness conflates refusal with unfaithfulness.** "المعلومة غير متوفرة في النصوص" is not
"supported by the reference texts", so the judge scores refusals near-randomly — in your committed
run, 19 of 35 refusals were called faithful and 16 unfaithful, a coin flip on identical behaviour.
Reported both including and excluding refusals now.

**9. Smaller things.** `torch_dtype=` is deprecated — replaced with a post-load cast that works on
every version. `truncation_side` set to `"left"` so an overflowing prompt drops context rather than
the trailing output-format instruction, and truncation is counted rather than silent.

## One thing I did not "fix", because it is your call

Cell 24 labels `C1_msa_e5 − C2_darija_e5` as *"replicates the earlier n=200 result"*, and the
header calls C2 the *"same seed as the earlier n=200 run, for comparison"*. **The seed matches but
the prompt does not.** This notebook's `GEN_PROMPT` adds a constraint the earlier run did not have:

> `أجب بالعربية فقط. ممنوع استعمال أي كلمة بحرف لاتيني.`

That is a deliberate change — and one that targets a failure visible in the earlier run, which
produced answers like `-Smithiya الحي لي استقرو…`. But it means C2 here is **not** the same
condition as `C2_darija_base` there, so this is not a replication, and the two C2 numbers should
not be differenced. The label now says "same retrieval, different prompt — not a replication".
Set the prompt back to the earlier wording if you want a true replication arm.

### Install

In [ ]:
import importlib.util, subprocess, sys

need = [p for p, m in [("rank_bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"),
                       ("transformers", "transformers"), ("accelerate", "accelerate")]
        if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)

import torch
# bitsandbytes only matters on CUDA; skip it elsewhere so this stays runnable on CPU.
if torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=False)
print("deps ready")

### Paths

Resolves the repo's `data/` directory, then the working directory (what a Colab upload gives you),
then GitHub — and prints which it used. `corpus_v2.json` is committed here as `data/corpus.json`.

In [ ]:
import os, json, urllib.request
from pathlib import Path
import re, gc, random, time
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file on disk: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,
    "retrieve_k": 20,      # candidates passed to the reranker
    "final_k": 5,          # passages actually given to the generator, after reranking
    "reranker": "BAAI/bge-reranker-v2-m3",

    # The earlier run's raw CSV, for the combined validation sheet. The original
    # pointed at "generation_n200_raw.csv", which does not exist anywhere in this
    # repo -- so the "combined" sheet silently covered only one run.
    "prior_run_csv": "generation_local_raw.csv",

    "n_eval": 200,
    "llm": "Qwen/Qwen2.5-7B-Instruct",
    "load_4bit": True,         # requires CUDA + bitsandbytes; auto-disabled otherwise
    "max_prompt_tokens": 3072,
    "rerank_max_length": 512,
    "rerank_batch_size": 16,
    "max_new_tokens": 128,
    "judge_max_new_tokens": 40,
    "batch_size": 8,

    "sheet_n_per_run": 50,
    "checkpoint": "gen_reranked_checkpoint.json",
    "seed": 42,
}
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c)
print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | evaluating {len(eval_qa)} (same seed as the earlier n=200 run)")

### BM25 + Arabic normalization

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Stage 1: retrieve top-K candidates

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Building retrieval index...")
bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve_candidates(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = CONFIG["alpha"] * minmax(corpus_emb @ q) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(query))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

K = CONFIG["retrieve_k"]
raw_candidates = {}
for field in ["msa_query", "darija_query"]:
    raw_candidates[field] = {q["id"]: retrieve_candidates(q[field], K) for q in eval_qa}
    print(f"  {field}: top-{K} retrieved")

del bi, corpus_emb
free_mem()

### Stage 2: rerank the Darija candidates, then free the reranker

In [ ]:
from sentence_transformers import CrossEncoder

print(f"Loading reranker: {CONFIG['reranker']}")
# Cast after loading rather than automodel_args={"torch_dtype": ...}: that kwarg is
# deprecated in new transformers and absent in old ones.
ce = CrossEncoder(CONFIG["reranker"], max_length=CONFIG["rerank_max_length"], trust_remote_code=True)
try:
    ce.model = ce.model.to(dtype=torch.float32)
except Exception:
    pass

reranked_darija = {}
for q in eval_qa:
    cands = raw_candidates["darija_query"][q["id"]]
    pairs = [(q["darija_query"], corpus_map[c]) for c in cands]
    scores = ce.predict(pairs, batch_size=CONFIG["rerank_batch_size"], show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    reranked_darija[q["id"]] = [cands[i] for i in order]

del ce
free_mem()
print("Reranking complete.")

### Build final top-k contexts for all four conditions

In [ ]:
FK = CONFIG["final_k"]
contexts = {
    "C1_msa_e5":          {q["id"]: raw_candidates["msa_query"][q["id"]][:FK] for q in eval_qa},
    "C2_darija_e5":       {q["id"]: raw_candidates["darija_query"][q["id"]][:FK] for q in eval_qa},
    "C5_darija_reranked": {q["id"]: reranked_darija[q["id"]][:FK] for q in eval_qa},
    "C4_oracle":          {q["id"]: [q["source_chunk_id"]] for q in eval_qa},
}

for cond, d in contexts.items():
    hit = np.mean([q["source_chunk_id"] in d[q["id"]] for q in eval_qa])
    print(f"  {cond:<22} gold in top-{FK}: {hit:.3f}")

# The whole point of the notebook, visible before any generation runs:
g2 = np.mean([q["source_chunk_id"] in contexts["C2_darija_e5"][q["id"]] for q in eval_qa])
g5 = np.mean([q["source_chunk_id"] in contexts["C5_darija_reranked"][q["id"]] for q in eval_qa])
print(f"\nRetrieval gain from reranking (gold in top-{FK}): {g2:.3f} -> {g5:.3f}  ({g5-g2:+.3f})")
print("Whether that reaches the ANSWER is what the generation below measures.")

### Load the LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

use_4bit = bool(CONFIG["load_4bit"]) and torch.cuda.is_available()
quant = None
if use_4bit:
    try:
        from transformers import BitsAndBytesConfig
        quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    except Exception as e:
        print(f"4-bit unavailable ({type(e).__name__}); loading unquantised.")
elif CONFIG["load_4bit"]:
    print("load_4bit requested but CUDA is absent - bitsandbytes has no CPU/TPU backend, "
          "so the model loads unquantised.")

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"      # required for correct batched generation
tok.truncation_side = "left"   # overflow must drop CONTEXT, never the trailing instruction

kw = {"quantization_config": quant} if quant else {}
if torch.cuda.is_available():
    kw["device_map"] = "auto"
llm = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], **kw)
if not quant:
    llm = llm.to(dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
llm.eval()
print(f"Loaded {CONFIG['llm']}  (4-bit: {bool(quant)})")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - this will be slow")

TRUNCATED = 0

@torch.no_grad()
def chat_batch(prompts, max_new_tokens):
    global TRUNCATED
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    for t in texts:
        if len(tok(t)["input_ids"]) > CONFIG["max_prompt_tokens"]:
            TRUNCATED += 1
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=CONFIG["max_prompt_tokens"]).to(llm.device)
    out_ = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                        pad_token_id=tok.pad_token_id)
    gen_ = out_[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen_]

print("Smoke test:", chat_batch(["\u0623\u062c\u0628 \u0628\u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629: \u0645\u0627 \u0639\u0627\u0635\u0645\u0629 \u0627\u0644\u0645\u063a\u0631\u0628\u061f"], 20)[0])

### Prompts and judge parsing

Note the Arabic-only constraint in `GEN_PROMPT`. It is **not** in the earlier n=200 run's prompt,
so C2 here is not a replication of that run's C2 — see the note at the top.

`parse_judge` is the corrected version: fields are searched independently instead of line by line,
whichever of yes/no appears first wins, a reply that merely echoes the template counts as unparsed,
and `judge_parsed` makes malformed replies visible instead of scoring them (0, 0).

In [ ]:
GEN_PROMPT = """\u0623\u062c\u0628 \u0639\u0646 \u0627\u0644\u0633\u0624\u0627\u0644 \u0627\u0644\u062a\u0627\u0644\u064a \u0627\u0639\u062a\u0645\u0627\u062f\u0627 \u0641\u0642\u0637 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u0641\u0642\u0629.

\u0642\u0648\u0627\u0639\u062f \u0625\u0644\u0632\u0627\u0645\u064a\u0629:
- \u0623\u062c\u0628 \u0628\u0627\u0644\u0639\u0631\u0628\u064a\u0629 \u0641\u0642\u0637. \u0645\u0645\u0646\u0648\u0639 \u0627\u0633\u062a\u0639\u0645\u0627\u0644 \u0623\u064a \u0643\u0644\u0645\u0629 \u0628\u062d\u0631\u0641 \u0644\u0627\u062a\u064a\u0646\u064a.
- \u0625\u0630\u0627 \u0644\u0645 \u062a\u0643\u0646 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0645\u0648\u062c\u0648\u062f\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635\u060c \u0627\u0643\u062a\u0628 \u0628\u0627\u0644\u0636\u0628\u0637: \u0627\u0644\u0645\u0639\u0644\u0648\u0645\u0629 \u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635
- \u0644\u0627 \u062a\u0633\u062a\u0639\u0645\u0644 \u0623\u064a \u0645\u0639\u0631\u0641\u0629 \u062e\u0627\u0631\u062c\u064a\u0629.
- \u0623\u062c\u0628 \u0628\u062c\u0645\u0644\u0629 \u0648\u0627\u062d\u062f\u0629 \u0642\u0635\u064a\u0631\u0629 \u0641\u0642\u0637.

\u0627\u0644\u0646\u0635\u0648\u0635:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}

\u0627\u0644\u0625\u062c\u0627\u0628\u0629:"""

JUDGE_PROMPT = """\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629: {gold}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629: {answer}

\u0623\u062c\u0628 \u0639\u0646 \u0633\u0624\u0627\u0644\u064a\u0646 \u0628\u062f\u0642\u0629:
1. \u0647\u0644 \u0643\u0644 \u0645\u0627 \u0648\u0631\u062f \u0641\u064a \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u062f\u0639\u0648\u0645 \u0635\u0631\u0627\u062d\u0629 \u0628\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629\u061f
2. \u0647\u0644 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u0637\u0627\u0628\u0642\u0629 \u0641\u064a \u0627\u0644\u0645\u0639\u0646\u0649 \u0644\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629\u061f \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0635\u064a\u0627\u063a\u0629 \u0645\u0642\u0628\u0648\u0644\u060c \u0623\u0645\u0627 \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0623\u0631\u0642\u0627\u0645 \u0623\u0648 \u0627\u0644\u0623\u0633\u0645\u0627\u0621 \u0623\u0648 \u0627\u0644\u062a\u0648\u0627\u0631\u064a\u062e \u0641\u063a\u064a\u0631 \u0645\u0642\u0628\u0648\u0644.

\u0623\u062c\u0628 \u0628\u0647\u0630\u0627 \u0627\u0644\u0634\u0643\u0644 \u0641\u0642\u0637 \u0648\u0628\u062f\u0648\u0646 \u0623\u064a \u0634\u0631\u062d:
\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627
\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627"""

REFUSAL = "\u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629"

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

_YES, _NO = "\u0646\u0639\u0645", "\u0644\u0627"
def _verdict(text, field):
    """Value of one labelled field, wherever it appears. None if absent or unusable."""
    m = re.search(field + r"\s*[:\uFF1A]?\s*([^\n\u060c,]*)", text)
    if not m:
        return None
    v = m.group(1).strip()
    if re.fullmatch(_YES + r"\s*/\s*" + _NO, v):   # echoed the instruction verbatim
        return None
    iy, ino = v.find(_YES), v.find(_NO)
    if iy == -1 and ino == -1:
        return None
    if iy == -1:
        return 0
    if ino == -1:
        return 1
    return 1 if iy < ino else 0

def parse_judge(text):
    """-> (faithful, correct, parsed)."""
    t = (text or "").replace("\u060c", " ")
    f = _verdict(t, "\u0645\u062f\u0639\u0648\u0645")
    c = _verdict(t, "\u0645\u0637\u0627\u0628\u0642")
    return (f or 0), (c or 0), int(f is not None and c is not None)

_c = [("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (0, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645 \u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),   # the original bug
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\u060c \u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627", (0, 0, 0)),
      ("", (0, 0, 0)),
      ("\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0635\u062d\u064a\u062d\u0629", (0, 0, 0))]
for txt, exp in _c:
    got = parse_judge(txt)
    assert got == exp, f"parse_judge regression: {txt!r} -> {got}, expected {exp}"
print("parse_judge: all 8 regression cases pass (incl. both verdicts on one line)")

CONDITION_QUERY = {
    "C1_msa_e5": "msa_query", "C2_darija_e5": "darija_query",
    "C5_darija_reranked": "darija_query", "C4_oracle": "darija_query",
}

### Run generation + judging (batched + checkpointed)

In [ ]:
from tqdm.auto import tqdm

CKPT = out(CONFIG["checkpoint"])
records = []
if os.path.exists(CKPT):
    records = json.load(open(CKPT, encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["condition"]) for r in records}
byid = {q["id"]: q for q in eval_qa}
B = CONFIG["batch_size"]

for cond, ctx_map in contexts.items():
    qfield = CONDITION_QUERY[cond]
    todo = [q for q in eval_qa if (q["id"], cond) not in done]
    if not todo:
        continue
    print(f"\n=== {cond} ({len(todo)} to do) ===")
    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [ctx_map[q["id"]] for q in batch]
        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q[qfield])
                              for q, c in zip(batch, chunks)], CONFIG["max_new_tokens"])
        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c), question=q["msa_query"],
                                                   gold=q["gold_answer"], answer=a)
                               for q, c, a in zip(batch, chunks, answers)],
                              CONFIG["judge_max_new_tokens"])
        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok, parsed = parse_judge(v)
            records.append({
                "qid": q["id"], "condition": cond,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "answer": a, "faithful": f, "correct": ok,
                "judge_parsed": parsed, "judge_raw": v,
                "refused": int(REFUSAL in (a or "")),
            })
        json.dump(records, open(CKPT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv(out("generation_reranked_raw.csv"), index=False)
print(f"\nComplete: {len(gen)} generations.")

# Both of these were silent in the original.
pf = 1 - gen.judge_parsed.mean()
print(f"Judge parse-failure rate: {pf:.1%}"
      + ("  <- metrics below are understated by this much" if pf > 0.01 else "  (negligible)"))
print(f"Prompts truncated: {TRUNCATED}" + ("  <- raise the prompt length" if TRUNCATED else "  (none)"))

### Results: does the reranker's retrieval gain reach the answers?

In [ ]:
print("=" * 90)
print("RESULTS BY CONDITION")
print("=" * 90)
order = [c for c in CONDITION_QUERY if c in gen.condition.unique()]
summary = gen.groupby("condition").agg(
    n=("qid", "count"), gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"), correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"), judge_parsed=("judge_parsed", "mean"),
).reindex(order)

# A refusal is not "supported by the reference texts", so the judge scores refusals
# near-randomly. Reporting both keeps the headline from being driven by refusal rate.
ans = gen[gen.refused == 0]
summary["faithfulness_answered"] = ans.groupby("condition")["faithful"].mean().reindex(order)
summary["correctness_answered"] = ans.groupby("condition")["correct"].mean().reindex(order)

print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv(out("generation_reranked_summary.csv"))
print("""
  *_answered  same metric excluding refusals -- see the note in the code above
  judge_parsed  share of judge replies that parsed (1.000 = all good)
""")

### The connecting comparison, with paired bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_cond, b_cond, col):
    a = gen[gen.condition == a_cond].set_index("qid")[col]
    b = gen[gen.condition == b_cond].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 90)
print("DOES THE RERANKER'S RETRIEVAL GAIN REACH THE FINAL ANSWERS?")
print("=" * 90)
comps = [
    # NOT labelled a replication: this notebook's GEN_PROMPT adds an Arabic-only
    # constraint the earlier n=200 run did not have, so its C2 and this C2 are
    # different conditions and must not be differenced across notebooks.
    ("C1_msa_e5", "C2_darija_e5", "Dialect gap WITHOUT reranking (same retrieval as the earlier run, different prompt - not a replication)"),
    ("C5_darija_reranked", "C2_darija_e5", "Does reranking improve the ANSWER, not just the ranking?"),
    ("C1_msa_e5", "C5_darija_reranked", "Residual gap AFTER reranking"),
]
rows = []
for a, b, label in comps:
    if a not in gen.condition.unique() or b not in gen.condition.unique():
        continue
    print(f"\n{label}\n   [{a} - {b}]")
    for col in ["correct", "faithful"]:
        d, lo, hi = paired(a, b, col)
        sig = "yes" if (lo > 0 or hi < 0) else "no"
        print(f"  {col:<10} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  significant: {sig}")
        rows.append({"comparison": f"{a} vs {b}", "metric": col, "diff": d, "lo": lo, "hi": hi,
                     "significant": sig})
pd.DataFrame(rows).to_csv(out("generation_reranked_comparisons.csv"), index=False)

### Build the COMBINED correctness-validation sheet

Three fixes here. The prior-run CSV now resolves to the file that actually exists
(`results/generation_local_raw.csv`) instead of a name that is nowhere in the repo, so the sheet
really does cover both rounds. The sample now reaches the requested size instead of stopping at
`n // n_conditions * n_conditions`. And rows whose qid cannot be resolved — previously emitted with
every field blank, unlabelable and unflagged — are counted and dropped.

In [ ]:
print("\n" + "=" * 90)
print("BUILDING COMBINED JUDGE-VALIDATION SHEET")
print("=" * 90)

def stratified_sample(df, n, qa_lookup, run_label):
    """Balanced across conditions and across the judge's own verdict, topped up to n."""
    conds = df["condition"].nunique()
    n_per = max(1, n // conds)
    parts = []
    for cond, grp in df.groupby("condition"):
        wrong, right = grp[grp["correct"] == 0], grp[grp["correct"] == 1]
        n_wrong = min(len(wrong), max(1, n_per // 2))
        n_right = min(len(right), n_per - n_wrong)
        parts.append(pd.concat([
            wrong.sample(n_wrong, random_state=42) if n_wrong else wrong.head(0),
            right.sample(n_right, random_state=42) if n_right else right.head(0),
        ]))
    sample = pd.concat(parts)

    # n // conds * conds < n whenever n is not divisible by the condition count:
    # 50 // 4 * 4 = 48. Top up from whatever is left so the sheet is the size asked for.
    if len(sample) < n:
        rest = df.drop(index=sample.index, errors="ignore")
        if len(rest):
            sample = pd.concat([sample, rest.sample(min(n - len(sample), len(rest)), random_state=42)])
    sample = sample.sample(frac=1, random_state=42).reset_index(drop=True).head(n)

    rows, missing = [], 0
    for _, r in sample.iterrows():
        q = qa_lookup.get(r["qid"])
        if not q:
            missing += 1          # would otherwise become an all-blank, unlabelable row
            continue
        rows.append({
            "run": run_label, "qid": r["qid"], "condition": r["condition"],
            "msa_query": q.get("msa_query", ""), "darija_query": q.get("darija_query", ""),
            "gold_answer": q.get("gold_answer", ""), "model_answer": r.get("answer", ""),
            "llm_correct": int(r["correct"]), "human_correct": "",
        })
    if missing:
        print(f"  WARNING [{run_label}]: {missing} rows dropped - qid not in the evaluation set.")
    return pd.DataFrame(rows)

# Built from the FULL benchmark, not just this run's eval_qa. The prior run may have
# used a different n_eval or split, so its qids need not be a subset of this run's --
# and an unresolved qid silently became an all-blank, unlabelable row before.
qa_lookup = {q["id"]: q for q in wiki_qa}
N_SHEET = CONFIG["sheet_n_per_run"]
sheets = [stratified_sample(gen, N_SHEET, qa_lookup, "reranked_run")]

prior_path = find_file(CONFIG["prior_run_csv"], "generation_n200_raw.csv")
if prior_path:
    prior = pd.read_csv(prior_path)
    sheets.append(stratified_sample(prior, N_SHEET, qa_lookup, "prior_run"))
    print(f"  Prior run: {prior_path} ({len(prior)} rows, "
          f"conditions: {', '.join(sorted(prior.condition.unique()))})")
else:
    print(f"  Prior run CSV not found (looked for {CONFIG['prior_run_csv']}).")
    print("  Sheet covers only this run.")

combined = pd.concat(sheets, ignore_index=True)
sheet_path = out("combined_labeling_sheet.csv")
combined.to_csv(sheet_path, index=False, encoding="utf-8-sig")

print(f"\nWrote {len(combined)} rows to {sheet_path}")
for label, cnt in combined["run"].value_counts().items():
    print(f"  {label:<14} {cnt}")

print("""
To use it: fill in `human_correct` with 0 or 1 for every row, judging the model_answer
against gold_answer yourself. Then compare that column with `llm_correct` -- agreement
rate, and the disagreements broken down by condition, tell you whether the LLM judge can
be trusted for the headline correctness numbers.

(The original cell pointed at `python validate_judge.py score --sheet ...`; there is no
validate_judge.py anywhere in this repo, so that instruction could not be followed.)
""")

# The original ended with a bare `from google.colab import files`, which raises outside Colab.
try:
    from google.colab import files
    for f in ["generation_reranked_raw.csv", "generation_reranked_summary.csv",
              "generation_reranked_comparisons.csv", "combined_labeling_sheet.csv"]:
        files.download(out(f))
except ImportError:
    print(f"(Not in Colab - all outputs are on disk under {OUT_DIR.resolve()})")